In [1]:
import os, itertools
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

FILES, LABEL_PATHS = {}, []
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if f.endswith(".npz"):
            continue
        FILES.setdefault(f, os.path.join(root, f))
        if f == "report_labels.csv":
            LABEL_PATHS.append(os.path.join(root, f))
print(f"Indeks: {len(FILES)} file")

LABEL_PATH = [p for p in LABEL_PATHS if "label-ensemble" in p]
assert LABEL_PATH, f"Tambahkan input rsna-knee-01d-label-ensemble. Ditemukan: {LABEL_PATHS}"
LABEL_PATH = LABEL_PATH[0]
print("Label acuan:", LABEL_PATH)

def mean_auc(y, p):
    """AUC per kelas. Label 0.5 dan kosong diabaikan; label lunak dibinerkan di 0.5."""
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk, pk = (y[ok, k] > 0.5).astype(float), p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs))

def ens(combo, P):
    return np.mean([P[t] for t in combo], axis=0)

def boot_diff(pa, pb, y, n=1000, seed=0):
    """Selisih AUC (pb - pa) dengan bootstrap; pa dan pb sudah berupa matriks prediksi."""
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        d.append(mean_auc(y[i], pb[i]) - mean_auc(y[i], pa[i]))
    return np.nanmean(d), np.nanpercentile(d, 2.5), np.nanpercentile(d, 97.5)

Indeks: 126 file
Label acuan: /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv


In [2]:
D = ["effb0", "dinov2s_student_v1d", "dinov2s_student_v2", "dinov2s_student_v3", "effb0_student_v3",
     "dinov2s_student_d32", "dinov2s_student_d32_s7", "effb0_student_d32_s7"]          # LB 0,887
NEW = ["dinov2b_student_v1d", "dinov2s_student_v1d_s7", "dinov2s_student_v1d_s11"]
NEED = D + NEW

val, gold = {}, {}
for t in NEED:
    val[t] = pd.read_csv(FILES[f"val_preds_{t}.csv"])
    gold[t] = pd.read_csv(FILES[f"gold_preds_{t}.csv"])
print("Model dimuat:", len(NEED))

rep = pd.read_csv(LABEL_PATH)
tr = pd.read_csv(f"{COMP}/train.csv")
gold_df = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]

val_ids = val[NEED[0]]["StudyInstanceUID"]
gold_ids = gold[NEED[0]]["StudyInstanceUID"]
for t in NEED:
    assert set(val[t]["StudyInstanceUID"]) == set(val_ids), f"split val {t} berbeda"
    assert set(gold[t]["StudyInstanceUID"]) == set(gold_ids), f"studi gold {t} berbeda"

y_val = rep.set_index("StudyInstanceUID").loc[val_ids, LABELS].values.astype(float)
y_gold = gold_df.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
P_val = {t: val[t].set_index("StudyInstanceUID").loc[val_ids, LABELS].values for t in NEED}
P_gold = {t: gold[t].set_index("StudyInstanceUID").loc[gold_ids, LABELS].values for t in NEED}
print(f"Studi val: {len(val_ids)} | studi gold: {len(gold_ids)}")

Model dimuat: 11
Studi val: 435 | studi gold: 58


In [3]:
print("=== Model tunggal (urut gold) ===")
single = pd.DataFrame([{"model": t, "val": mean_auc(y_val, P_val[t]),
                        "gold": mean_auc(y_gold, P_gold[t])} for t in NEED])
print(single.sort_values("gold", ascending=False).round(4).to_string(index=False))

rows = []
for r in range(4, len(NEED) + 1):
    for combo in itertools.combinations(NEED, r):
        rows.append({"combo": " + ".join(combo), "n": r,
                     "val": mean_auc(y_val, ens(combo, P_val)), "gold": mean_auc(y_gold, ens(combo, P_gold))})
res = pd.DataFrame(rows); res["rata2"] = (res["val"] + res["gold"]) / 2
print(f"\nKombinasi diuji: {len(res)}")
print("\n=== 10 kombinasi terbaik (rata-rata val dan gold) ===")
print(res.sort_values("rata2", ascending=False).head(10).round(4).to_string(index=False))

best = res.sort_values("rata2", ascending=False).iloc[0]["combo"].split(" + ")
cands = {
    "D (LB 0,887)": D,
    "D + 2 seed DINOv2-S v1d": D + ["dinov2s_student_v1d_s7", "dinov2s_student_v1d_s11"],
    "D + DINOv2-Base": D + ["dinov2b_student_v1d"],
    "D + 3 model baru": D + NEW,
    "Kombinasi terbaik": best,
}
print("\n=== Kandidat (selisih gold dibanding D, bootstrap 1.000x) ===")
pD = ens(D, P_gold)
for name, combo in cands.items():
    v, g = mean_auc(y_val, ens(combo, P_val)), mean_auc(y_gold, ens(combo, P_gold))
    if name.startswith("D ("):
        print(f"{name:26s} val {v:.4f} | gold {g:.4f} | (acuan)")
    else:
        m, lo, hi = boot_diff(pD, ens(combo, P_gold), y_gold)
        print(f"{name:26s} val {v:.4f} | gold {g:.4f} | selisih {m:+.4f} (95% CI {lo:+.4f} .. {hi:+.4f})")
print("\nIsi kombinasi terbaik:", best)

=== Model tunggal (urut gold) ===
                  model    val   gold
    dinov2s_student_v1d 0.8753 0.8610
 dinov2s_student_d32_s7 0.8715 0.8604
dinov2s_student_v1d_s11 0.8741 0.8569
    dinov2b_student_v1d 0.8719 0.8543
     dinov2s_student_v3 0.8721 0.8538
 dinov2s_student_v1d_s7 0.8745 0.8516
    dinov2s_student_d32 0.8770 0.8493
       effb0_student_v3 0.8683 0.8484
   effb0_student_d32_s7 0.8414 0.8479
     dinov2s_student_v2 0.8468 0.8436
                  effb0 0.8289 0.8383

Kombinasi diuji: 1816

=== 10 kombinasi terbaik (rata-rata val dan gold) ===
                                                                                                                                                                                   combo  n    val   gold  rata2
                                           effb0 + dinov2s_student_v1d + dinov2s_student_v2 + effb0_student_v3 + effb0_student_d32_s7 + dinov2s_student_v1d_s7 + dinov2s_student_v1d_s11  7 0.8809 0.8724 0.8766
              

In [4]:
# 1) Apakah data uji punya laporan?
test = pd.read_csv(f"{COMP}/test.csv")
print("Kolom test.csv:", test.columns.tolist())
print("Jumlah studi uji (yang terlihat):", len(test))
if "Report" in test.columns:
    print("Studi uji yang punya laporan:", int(test["Report"].notna().sum()))
else:
    print("test.csv TIDAK punya kolom Report")

# 2) Gabungan prediksi citra (ensemble D) dan label LLM pada 58 studi gold
L_gold = rep.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
L_gold = np.nan_to_num(L_gold, nan=0.5)
pg = ens(D, P_gold)
print(f"\nCitra saja (D):   gold {mean_auc(y_gold, pg):.4f}")
print(f"Label LLM saja:   gold {mean_auc(y_gold, L_gold):.4f}")
print("\nBobot laporan w  ->  gold AUC   (prediksi = (1-w) x citra + w x label LLM)")
for w in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    print(f"  w = {w:.1f}  ->  {mean_auc(y_gold, (1 - w) * pg + w * L_gold):.4f}")

# 3) Bootstrap: gabungan w=0,3 dibanding citra saja
fused = 0.7 * pg + 0.3 * L_gold
m, lo, hi = boot_diff(pg, fused, y_gold)
print(f"\nSelisih (w=0,3 - citra saja): {m:+.4f} (95% CI {lo:+.4f} .. {hi:+.4f})")

# 4) Per kelainan untuk w=0,3
print("\nPer kelainan (citra -> gabungan w=0,3):")
for k, l in enumerate(LABELS):
    ok = ~np.isnan(y_gold[:, k]) & (y_gold[:, k] != 0.5)
    yk = (y_gold[ok, k] > 0.5).astype(float)
    if len(np.unique(yk)) == 2:
        print(f"  {l:18s} {roc_auc_score(yk, pg[ok, k]):.3f} -> {roc_auc_score(yk, fused[ok, k]):.3f}")

Kolom test.csv: ['StudyInstanceUID']
Jumlah studi uji (yang terlihat): 3
test.csv TIDAK punya kolom Report

Citra saja (D):   gold 0.8721
Label LLM saja:   gold 0.8465

Bobot laporan w  ->  gold AUC   (prediksi = (1-w) x citra + w x label LLM)
  w = 0.1  ->  0.8892
  w = 0.2  ->  0.9010
  w = 0.3  ->  0.9063
  w = 0.4  ->  0.9071
  w = 0.5  ->  0.9065
  w = 0.6  ->  0.9051

Selisih (w=0,3 - citra saja): +0.0343 (95% CI +0.0192 .. +0.0511)

Per kelainan (citra -> gabungan w=0,3):
  ACL                0.897 -> 0.934
  MCL                0.893 -> 0.946
  Medial Meniscus    0.869 -> 0.953
  Lateral Meniscus   0.773 -> 0.872
  Medial OA          0.989 -> 0.988
  Lateral OA         0.843 -> 0.878
  PF OA              0.794 -> 0.838
  Effusion           0.914 -> 0.917
  Synovitis          0.757 -> 0.793
  Baker's            0.973 -> 0.975
  Contusion          0.880 -> 0.897
  Fracture           0.882 -> 0.885


In [5]:
# Ensemble 5-fold DINOv2 (guru baru) sebagai satu anggota tambahan; penilaian hanya di gold
kf_files = sorted(p for f, p in FILES.items() if f.startswith("gold_preds_dinov2s_kf_f"))
print("File fold:", len(kf_files))
g = pd.DataFrame({"StudyInstanceUID": gold_ids})
P_kf = np.mean([g.merge(pd.read_csv(p), on="StudyInstanceUID")[LABELS].values for p in kf_files], axis=0)

pD = ens(D, P_gold)
cands = {
    "D + DINOv2 5-fold (bobot 1)": (pD * len(D) + P_kf) / (len(D) + 1),
    "D + DINOv2 5-fold (bobot 2)": (pD * len(D) + 2 * P_kf) / (len(D) + 2),
    "D, DINOv2 v1d diganti 5-fold": ens([t for t in D if t != "dinov2s_student_v1d"], P_gold) * (len(D) - 1) / len(D) + P_kf / len(D),
    "DINOv2 5-fold saja": P_kf,
}
print(f"\nD (LB 0,887)                      gold {mean_auc(y_gold, pD):.4f} | (acuan)")
for name, p in cands.items():
    m, lo, hi = boot_diff(pD, p, y_gold)
    print(f"{name:33s} gold {mean_auc(y_gold, p):.4f} | selisih {m:+.4f} (95% CI {lo:+.4f} .. {hi:+.4f})")

File fold: 5

D (LB 0,887)                      gold 0.8721 | (acuan)
D + DINOv2 5-fold (bobot 1)       gold 0.8715 | selisih -0.0006 (95% CI -0.0033 .. +0.0019)
D + DINOv2 5-fold (bobot 2)       gold 0.8707 | selisih -0.0015 (95% CI -0.0058 .. +0.0025)
D, DINOv2 v1d diganti 5-fold      gold 0.8721 | selisih -0.0000 (95% CI -0.0036 .. +0.0033)
DINOv2 5-fold saja                gold 0.8658 | selisih -0.0064 (95% CI -0.0181 .. +0.0041)
